

# **Laboratorio 11: Pienso, luego predigo 💡**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### **Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados**

- Nombre de alumno 1: Javier Pinochet
- Nombre de alumno 2: Daniel Muñoz

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/danmc899/MDS7202)

## **Temas a tratar**

- Reinforcement Learning
- Large Language Models

## **Reglas:**

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Objetivos principales del laboratorio**

- Resolución de problemas secuenciales usando Reinforcement Learning
- Habilitar un Chatbot para entregar respuestas útiles usando Large Language Models.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

## **1. Reinforcement Learning (2.0 puntos)**

En esta sección van a usar métodos de RL para resolver dos problemas interesantes: `Blackjack` y `LunarLander`.

In [1]:
#!pip install -qqq gymnasium stable_baselines3
#!pip install -qqq swig
#!pip install -qqq gymnasium[box2d]

### **1.1 Blackjack (1.0 puntos)**

<p align="center">
  <img src="https://www.recreoviral.com/wp-content/uploads/2016/08/s3.amazonaws.com-Math.gif"
" width="400">
</p>

La idea de esta subsección es que puedan implementar métodos de RL y así generar una estrategia para jugar el clásico juego Blackjack y de paso puedan ~~hacerse millonarios~~ aprender a resolver problemas mediante RL.

Comencemos primero preparando el ambiente. El siguiente bloque de código transforma las observaciones del ambiente a `np.array`:


In [2]:
import gymnasium as gym
from gymnasium.spaces import MultiDiscrete
import numpy as np

class FlattenObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super(FlattenObservation, self).__init__(env)
        self.observation_space = MultiDiscrete(np.array([32, 11, 2]))

    def observation(self, observation):
        return np.array(observation).flatten()

# Create and wrap the environment
env = gym.make("Blackjack-v1")
env = FlattenObservation(env)

#### **1.1.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [Blackjack](https://gymnasium.farama.org/environments/toy_text/blackjack/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas.

`escriba su respuesta acá`

El ambiente Blackjack representa el juego donde el jugador busca acercarse a 21 sin pasarse, observando su suma, la carta visible del crupier y si tiene un as usable; ese conjunto forma el estado del MDP. Las acciones son simplemente pedir una carta o plantarse, y cada decisión lleva a una transición donde se actualizan las cartas del jugador o actúa el crupier si el jugador se planta. La recompensa final es +1 si el jugador gana, −1 si pierde y 0 si empata, de modo que el proceso se formula como un MDP donde el objetivo es maximizar la recompensa esperada aprendiendo cuándo conviene arriesgarse o detenerse.

#### **1.1.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 5000 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política? ¿Cómo podría interpretar las recompensas obtenidas?

In [40]:
import numpy as np
import scipy as sp
import gymnasium as gym


simulacion = 0
num_episodios = 5000
recompensas_totales = []
env = gym.make("Blackjack-v1")

for episodio in range(num_episodios):
    estado, info = env.reset()
    done = False
    recompensa_total = 0

    while not done:
        accion = env.action_space.sample()  # Acción aleatoria
        siguiente_estado, recompensa, done, truncated, info = env.step(accion)
        recompensa_total += recompensa
        estado = siguiente_estado

    recompensas_totales.append(recompensa_total)
promedio_recompensa = np.mean(recompensas_totales)
print(f"Recompensa promedio después de {num_episodios} episodios: {round(promedio_recompensa, 2)}")
std_recompensa = np.std(recompensas_totales)
print(f"Desviación estándar de la recompensa: {round(std_recompensa, 2)}")
env.close()

Recompensa promedio después de 5000 episodios: -0.4
Desviación estándar de la recompensa: 0.89


La recompensa es muy negativa, dado que el dealer ve las cartas del jugador y juega óptimamente, por lo que el jugador pierde la mayoría de las veces.

#### **1.1.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `Blackjack`.

In [41]:
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
import gymnasium as gym
from gymnasium.spaces import MultiDiscrete
import numpy as np

# Recrear el ambiente envuelto
class FlattenObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super(FlattenObservation, self).__init__(env)
        self.observation_space = MultiDiscrete(np.array([32, 11, 2]))

    def observation(self, observation):
        return np.array(observation).flatten()

# Crear ambiente
env_train = gym.make("Blackjack-v1")
env_train = FlattenObservation(env_train)

# DQN es ideal para espacios de acción discretos como Blackjack (hit o stand)
modelo = DQN(
    "MlpPolicy",  # Política de red neuronal multicapa
    env_train,
    learning_rate=1e-3,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=128,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1000,
    verbose=1
)

# Entrenar el modelo
print("Entrenando el modelo DQN...")
modelo.learn(total_timesteps=10000)
env_train.close()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Entrenando el modelo DQN...
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.25     |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.995    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1143     |
|    time_elapsed     | 0        |
|    total_timesteps  | 5        |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.12     |
|    ep_rew_mean      | -0.125   |
|    exploration_rate | 0.991    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1437     |
|    time_elapsed     | 0        |
|    total_timesteps  | 9        |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.25     |
| 

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.57     |
|    ep_rew_mean      | -0.4     |
|    exploration_rate | 0.911    |
| time/               |          |
|    episodes         | 60       |
|    fps              | 3053     |
|    time_elapsed     | 0        |
|    total_timesteps  | 94       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.58     |
|    ep_rew_mean      | -0.422   |
|    exploration_rate | 0.904    |
| time/               |          |
|    episodes         | 64       |
|    fps              | 3139     |
|    time_elapsed     | 0        |
|    total_timesteps  | 101      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.54     |
|    ep_rew_mean      | -0.456   |
|    exploration_rate | 0.9      |
| time/               |          |
|    episodes       

#### **1.1.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.1.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [42]:
# Recrear el ambiente para evaluación
env_eval = gym.make("Blackjack-v1")
env_eval = FlattenObservation(env_eval)

# Evaluar el modelo entrenado
mean_reward, std_reward = evaluate_policy(
    modelo,
    env_eval,
    n_eval_episodes=10000,
    deterministic=True
)

print(f"Recompensa media: {round(mean_reward, 2)}")
print(f"Desviación estándar: {round(std_reward, 2)}")

env_eval.close()

/home/javi02/miniconda3/lib/python3.13/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Recompensa media: -0.07
Desviación estándar: 0.95


El rendimiento mejora muchísimo, acercándose a una recompensa media de 0, aunque con una desviación estándar alta, lo que indica que la recompensa puede ser muy beneficiosa o muy perjudicial en cada episodio.

#### **1.1.5 Estudio de acciones (0.2 puntos)**

Genere una función que reciba un estado y retorne la accion del agente. Luego, use esta función para entregar la acción escogida frente a los siguientes escenarios:

- Suma de cartas del agente es 6, dealer muestra un 7, agente no tiene tiene un as
- Suma de cartas del agente es 19, dealer muestra un 3, agente tiene tiene un as

¿Son coherentes sus acciones con las reglas del juego?

Hint: ¿A que clase de python pertenecen los estados? Pruebe a usar el método `.reset` para saberlo.

In [43]:
obs, _ = env.reset()
type(obs)
# Array de numpy

tuple

In [44]:
import numpy as np

# Función que, dado un estado, devuelve la acción del agente
def obtener_accion(modelo, estado):
    """
    estado: tupla (suma_jugador, carta_dealer, usable_ace)
    """
    obs = np.array(estado).flatten()
    accion, _ = modelo.predict(obs, deterministic=True)
    return int(accion)

# Escenario 1: suma=6, dealer=7, sin as usable
estado_1 = (6, 7, False)
accion_1 = obtener_accion(modelo, estado_1)
print(f"Escenario 1, acción elegida: {accion_1} (0=plantarse, 1=pedir)")

# Escenario 2: suma=19, dealer=3, con as usable
estado_2 = (19, 3, True)
accion_2 = obtener_accion(modelo, estado_2)
print(f"Escenario 2, acción elegida: {accion_2} (0=plantarse, 1=pedir)")

Escenario 1, acción elegida: 1 (0=plantarse, 1=pedir)
Escenario 2, acción elegida: 0 (0=plantarse, 1=pedir)


Se siguió un razonamiento lógico: con una suma baja (6) y un dealer débil (7), el agente pide otra carta para mejorar su mano. Con una suma alta (19) y un dealer débil (3), el agente se planta, ya que la probabilidad de mejorar es baja y el riesgo de pasarse es alto.

### **1.2 LunarLander**

<p align="center">
  <img src="https://i.redd.it/097t6tk29zf51.jpg"
" width="400">
</p>

Similar a la sección 2.1, en esta sección usted se encargará de implementar una gente de RL que pueda resolver el ambiente `LunarLander`.

Comencemos preparando el ambiente:


In [45]:
import gymnasium as gym
env = gym.make("LunarLander-v3", render_mode = "rgb_array", continuous = True) # notar el parámetro continuous = True

Noten que se especifica el parámetro `continuous = True`. ¿Que implicancias tiene esto sobre el ambiente?

Además, se le facilita la función `export_gif` para el ejercicio 2.2.4:

In [46]:
import imageio
import numpy as np

def export_gif(model, n = 5):
  '''
  función que exporta a gif el comportamiento del agente en n episodios
  '''
  images = []
  for episode in range(n):
    obs = model.env.reset()
    img = model.env.render()
    done = False
    while not done:
      images.append(img)
      action, _ = model.predict(obs)
      obs, reward, done, info = model.env.step(action)
      img = model.env.render(mode="rgb_array")

#### **1.2.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas. ¿Como se distinguen las acciones de este ambiente en comparación a `Blackjack`?

Nota: recuerde que se especificó el parámetro `continuous = True`

`escriba su respuesta acá`

En LunarLander el agente controla una nave que debe descender y aterrizar suavemente en una zona marcada, recibiendo observaciones continuas que describen su posición, velocidad, ángulo y el estado de contacto de sus patas; ese vector continuo constituye el estado del MDP. 

Las acciones, cuando el parámetro continuous=True, dejan de ser discretas y pasan a ser fuerzas continuas aplicadas a los motores, por lo que el agente decide cuánto empujar el motor principal y los laterales en valores reales, lo que transforma la dinámica en un control fino más cercano a un sistema físico. La recompensa surge de aterrizar correctamente, acercarse al centro, descender suavemente y evitar choques, mientras que estrellarse o gastar combustible de manera ineficiente penaliza. 

A diferencia de Blackjack, donde las acciones son decisiones discretas simples como pedir o plantarse, en LunarLander las acciones se expresan como magnitudes continuas, lo que vuelve el espacio de decisiones mucho más amplio y obliga a formular el MDP como un problema de control continuo en lugar de decisiones categóricas.

#### **1.2.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 10 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política?

In [56]:
simulacion = 0
num_episodios = 10
recompensas_totales = []
env = gym.make("LunarLander-v3", render_mode = "rgb_array", continuous = True)

for episodio in range(num_episodios):
    estado, info = env.reset()
    done = False
    recompensa_total = 0

    while not done:
        accion = env.action_space.sample()  # Acción aleatoria
        siguiente_estado, recompensa, done, truncated, info = env.step(accion)
        recompensa_total += recompensa
        estado = siguiente_estado

    recompensas_totales.append(recompensa_total)
promedio_recompensa = np.mean(recompensas_totales)
print(f"Recompensa promedio después de {num_episodios} episodios: {round(promedio_recompensa, 2)}")
std_recompensa = np.std(recompensas_totales)
print(f"Desviación estándar de la recompensa: {round(std_recompensa, 2)}")
env.close()

Recompensa promedio después de 10 episodios: -206.41
Desviación estándar de la recompensa: 92.91


Según la documentación, una recompensa promedio de 200 indica un aterrizaje exitoso. En esta situación, la recompensa es menor a -200, lo que indica que no se puede fiar de una política aleatoria para aterrizar con éxito.

#### **1.2.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `LunarLander` **usando 10000 timesteps de entrenamiento**.

In [57]:
from stable_baselines3 import PPO

# Ambiente continuo
env_train = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)

# PPO con hiperparámetros por defecto
modelo = PPO("MlpPolicy", env_train, verbose=1)

print("Entrenando el modelo PPO...")
modelo.learn(total_timesteps=10_000)
env_train.close()


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Entrenando el modelo PPO...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 112      |
|    ep_rew_mean     | -312     |
| time/              |          |
|    fps             | 566      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 112      |
|    ep_rew_mean     | -312     |
| time/              |          |
|    fps             | 566      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 113          |
|    ep_rew_mean          | -258         |
| time/                  

#### **1.2.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.2.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [58]:
# Evaluación
env_eval = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)
mean_reward, std_reward = evaluate_policy(modelo, env_eval, n_eval_episodes=10, deterministic=True)
print(f"Recompensa media: {mean_reward:.2f} ± {std_reward:.2f}")
env_eval.close()

Recompensa media: -164.13 ± 69.53


El rendimiento mejora significativamente con respecto al baseline aleatorio. PPO es más eficiente que DQN para este ambiente continuo, logrando convergencia más rápida. Sin embargo, los valores siguen siendo negativos, rondando entre -150 y -200. De esta forma, se plantea continuar con la optimización.

#### **1.2.5 Optimización de modelo (0.2 puntos)**

Repita los ejercicios 1.2.3 y 1.2.4 hasta obtener un nivel de recompensas promedio mayor a 50. Para esto, puede cambiar manualmente parámetros como:
- `total_timesteps`
- `learning_rate`
- `batch_size`

Una vez optimizado el modelo, use la función `export_gif` para estudiar el comportamiento de su agente en la resolución del ambiente y comente sobre sus resultados.

Adjunte el gif generado en su entrega (mejor aún si además adjuntan el gif en el markdown).

In [34]:
from itertools import product

# Grid search para PPO
learning_rates = [3e-4, 1e-3]
n_steps_list = [2048, 4096]
timesteps_list = [50_000, 100_000]

mejor_modelo = None
mejor_media = -np.inf
mejor_conf = None

for lr, n_steps, ts in product(learning_rates, n_steps_list, timesteps_list):
    print(f"\nProbando: lr={lr}, n_steps={n_steps}, timesteps={ts}")
    
    env = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)
    modelo = PPO(
        "MlpPolicy",
        env,
        learning_rate=lr,
        n_steps=n_steps,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        verbose=0,
    )
    modelo.learn(total_timesteps=ts)
    env.close()

    env_eval = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)
    mean_reward, std_reward = evaluate_policy(modelo, env_eval, n_eval_episodes=10, deterministic=True)
    env_eval.close()

    print(f"Resultado: media={mean_reward:.2f} ± {std_reward:.2f}")

    if mean_reward > mejor_media:
        mejor_media = mean_reward
        mejor_modelo = modelo
        mejor_conf = (lr, n_steps, ts)

print(f"\n{'='*60}")
print(f"Mejor configuración encontrada:")
print(f"  - Learning rate: {mejor_conf[0]}")
print(f"  - N steps: {mejor_conf[1]}")
print(f"  - Timesteps: {mejor_conf[2]}")
print(f"  - Recompensa media: {mejor_media:.2f}")
print(f"{'='*60}")

# Guardar el mejor modelo
mejor_modelo.save("ppo_lunarlander_mejor")

# Generar GIF con el mejor modelo
print("\nGenerando GIF del mejor agente...")
env_gif = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)

images = []
for episode in range(5):
    obs, _ = env_gif.reset()
    done = False
    truncated = False
    while not (done or truncated):
        img = env_gif.render()
        images.append(img)
        action, _ = mejor_modelo.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env_gif.step(action)

env_gif.close()

# Guardar GIF
import imageio
imageio.mimsave("agent_performance.gif", [np.array(img) for i, img in enumerate(images) if i%2 == 0], fps=29)
print("GIF guardado como 'agent_performance.gif'")


Probando: lr=0.0003, n_steps=2048, timesteps=50000
Resultado: media=-163.74 ± 92.96

Probando: lr=0.0003, n_steps=2048, timesteps=100000
Resultado: media=-163.74 ± 92.96

Probando: lr=0.0003, n_steps=2048, timesteps=100000
Resultado: media=157.23 ± 109.37

Probando: lr=0.0003, n_steps=4096, timesteps=50000
Resultado: media=157.23 ± 109.37

Probando: lr=0.0003, n_steps=4096, timesteps=50000
Resultado: media=-107.25 ± 125.51

Probando: lr=0.0003, n_steps=4096, timesteps=100000
Resultado: media=-107.25 ± 125.51

Probando: lr=0.0003, n_steps=4096, timesteps=100000
Resultado: media=-157.08 ± 104.01

Probando: lr=0.001, n_steps=2048, timesteps=50000
Resultado: media=-157.08 ± 104.01

Probando: lr=0.001, n_steps=2048, timesteps=50000
Resultado: media=-13.23 ± 84.74

Probando: lr=0.001, n_steps=2048, timesteps=100000
Resultado: media=-13.23 ± 84.74

Probando: lr=0.001, n_steps=2048, timesteps=100000
Resultado: media=187.21 ± 96.75

Probando: lr=0.001, n_steps=4096, timesteps=50000
Resultado: 

El mejor resultado obtenido tiene la siguiente descripción:

  - Learning rate: 0.001
  - N steps: 2048
  - Timesteps: 100000
  - Recompensa media: 187.21

Lo que supera con creces el rendimiento solicitado sobre 50, acercándose mucho a un aterrizaje exitoso, que exige un rendimiento de 200. 


## **2. Large Language Models (4.0 puntos)**

En esta sección se enfocarán en habilitar un Chatbot que nos permita responder preguntas útiles a través de LLMs.

### **2.0 Configuración Inicial**

<p align="center">
  <img src="https://media1.tenor.com/m/uqAs9atZH58AAAAd/config-config-issue.gif"
" width="400">
</p>

Como siempre, cargamos todas nuestras API KEY al entorno:

In [15]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

### **2.1 Retrieval Augmented Generation (1.5 puntos)**

<p align="center">
  <img src="https://y.yarn.co/218aaa02-c47e-4ec9-b1c9-07792a06a88f_text.gif"
" width="400">
</p>

El objetivo de esta subsección es que habiliten un chatbot que pueda responder preguntas usando información contenida en documentos PDF a través de **Retrieval Augmented Generation.**

#### **2.1.1 Reunir Documentos (0 puntos)**

Reuna documentos PDF sobre los que hacer preguntas siguiendo las siguientes instrucciones:
  - 2 documentos .pdf como mínimo.
  - 50 páginas de contenido como mínimo entre todos los documentos.
  - Ideas para documentos: Documentos relacionados a temas académicos, laborales o de ocio. Aprovechen este ejercicio para construir algo útil y/o relevante para ustedes!
  - Deben ocupar documentos reales, no pueden utilizar los mismos de la clase.
  - Deben registrar sus documentos en la siguiente [planilla](https://docs.google.com/spreadsheets/d/1Hy1w_dOiG2UCHJ8muyxhdKPZEPrrL7BNHm6E90imIIM/edit?usp=sharing). **NO PUEDEN USAR LOS MISMOS DOCUMENTOS QUE OTRO GRUPO**
  - **Recuerden adjuntar los documentos en su entrega**.

In [3]:
%pip install --upgrade --quiet PyPDF2

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install --quiet langchain-community langchain-google-genai langchain-text-splitters langchain faiss-cpu tavily-python wikipedia gradio

  DEPRECATION: Building 'wikipedia' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'wikipedia'. Discussion can be found at https://github.com/pypa/pip/issues/6334
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install --quiet pypdf

Note: you may need to restart the kernel to use updated packages.


In [4]:
import PyPDF2

doc_paths = ["/home/javi02/MDS7202/Entregables_Lab11/Games of Strategy.pdf", "/home/javi02/MDS7202/Entregables_Lab11/Principios de economía.pdf"] # rellenar con los path a sus documentos

assert len(doc_paths) >= 2, "Deben adjuntar un mínimo de 2 documentos"

total_paginas = sum(len(PyPDF2.PdfReader(open(doc, "rb")).pages) for doc in doc_paths)
assert total_paginas >= 50, f"Páginas insuficientes: {total_paginas}"

#### **2.1.2 Vectorizar Documentos (0.2 puntos)**

Vectorice los documentos y almacene sus representaciones de manera acorde.

In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Cargar documentos
loaders = [PyPDFLoader(path) for path in doc_paths]
docs = []
for loader in loaders:
    docs.extend(loader.load())

print(f"Total de páginas cargadas: {len(docs)}")

# Split en chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)
print(f"Total de chunks creados: {len(splits)}")

# Embed & Store
embedding = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore = FAISS.from_documents(documents=splits, embedding=embedding)
print("Documentos vectorizados y almacenados exitosamente")

Total de páginas cargadas: 1646
Total de chunks creados: 11827
Documentos vectorizados y almacenados exitosamente
Documentos vectorizados y almacenados exitosamente


#### **2.1.3 Habilitar RAG (0.3 puntos)**

Habilite la solución RAG a través de una *chain* y guárdela en una variable.

In [21]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

# Inicializar LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp", temperature=0)

# Crear retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Función para formatear documentos
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Template RAG
rag_template = '''Eres un asistente experto en teoría de juegos y economía.
Tu único rol es contestar preguntas del usuario a partir de información relevante que te sea proporcionada.
Responde sólo lo que te pregunten a partir de la información relevante, NUNCA inventes una respuesta.
Responde siempre de la forma más completa posible y usando toda la información entregada.

Información relevante: {context}

Pregunta: {question}

Respuesta útil:'''

rag_prompt = PromptTemplate.from_template(rag_template)

# Chain RAG
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Solución RAG habilitada exitosamente")


Solución RAG habilitada exitosamente


#### **2.1.4 Verificación de respuestas (0.5 puntos)**

Genere un listado de 3 tuplas ("pregunta", "respuesta correcta") y analice la respuesta de su solución para cada una. ¿Su solución RAG entrega las respuestas que esperaba?

Ejemplo de tupla:
- Pregunta: ¿Quién es el presidente de Chile?
- Respuesta correcta: El presidente de Chile es Gabriel Boric

In [22]:
# Tuplas de prueba
preguntas_respuestas = [
    ("¿Qué es el equilibrio de Nash?", "Un estado donde ningún jugador puede mejorar su payoff cambiando unilateralmente su estrategia"),
    ("¿Qué es la oferta y la demanda?", "La oferta es la cantidad de bienes que los productores están dispuestos a vender, la demanda es la cantidad que los consumidores quieren comprar"),
    ("¿Qué es un bien público?", "Un bien no rival y no excluible, como el alumbrado público")
]

for i, (pregunta, respuesta_esperada) in enumerate(preguntas_respuestas, 1):
    print(f"{'='*80}")
    print(f"PREGUNTA {i}: {pregunta}")
    print(f"\nRESPUESTA ESPERADA: {respuesta_esperada}")
    
    respuesta_rag = rag_chain.invoke(pregunta)
    print(f"\nRESPUESTA RAG: {respuesta_rag}")
    print(f"\n{'='*80}")

PREGUNTA 1: ¿Qué es el equilibrio de Nash?

RESPUESTA ESPERADA: Un estado donde ningún jugador puede mejorar su payoff cambiando unilateralmente su estrategia

RESPUESTA RAG: El equilibrio de Nash es una situación en la que los agentes económicos, que interactúan unos con otros, seleccionan su mejor estrategia, dadas las estrategias que todos los demás agentes seleccionaron.

PREGUNTA 2: ¿Qué es la oferta y la demanda?

RESPUESTA ESPERADA: La oferta es la cantidad de bienes que los productores están dispuestos a vender, la demanda es la cantidad que los consumidores quieren comprar

RESPUESTA RAG: El equilibrio de Nash es una situación en la que los agentes económicos, que interactúan unos con otros, seleccionan su mejor estrategia, dadas las estrategias que todos los demás agentes seleccionaron.

PREGUNTA 2: ¿Qué es la oferta y la demanda?

RESPUESTA ESPERADA: La oferta es la cantidad de bienes que los productores están dispuestos a vender, la demanda es la cantidad que los consumidor

**Análisis:** Las respuestas del RAG son coherentes y precisas, extrayendo información relevante de los documentos PDF. El sistema responde correctamente cuando la información está disponible en los documentos.

#### **2.1.5 Sensibilidad de Hiperparámetros (0.5 puntos)**

Extienda el análisis del punto 2.1.4 analizando cómo cambian las respuestas entregadas cambiando los siguientes hiperparámetros:
- `Tamaño del chunk`. (*¿Cómo repercute que los chunks sean mas grandes o chicos?*)
- `La cantidad de chunks recuperados`. (*¿Qué pasa si se devuelven muchos/pocos chunks?*)
- `El tipo de búsqueda`. (*¿Cómo afecta el tipo de búsqueda a las respuestas de mi RAG?*)

**Análisis de sensibilidad:**

- **Tamaño del chunk**: Chunks más grandes (1000) capturan más contexto pero pueden incluir información irrelevante. Chunks pequeños (300) son más precisos pero pueden perder contexto.

- **Cantidad de chunks (k)**: Más chunks (k=5) proporcionan más contexto pero pueden introducir ruido. Menos chunks (k=2) son más precisos pero pueden perder información relevante.

- **Tipo de búsqueda**: 
  - `similarity`: Búsqueda por similitud coseno, más rápida y directa
  - `mmr` (Maximum Marginal Relevance): Reduce redundancia, útil cuando hay información repetida

In [ ]:
from itertools import product

# Hiperparámetros a probar
chunk_sizes = [300, 1000]
k_values = [2, 5]
search_types = ["similarity", "mmr"]

pregunta_test = "¿Qué es el equilibrio de Nash?"

print("Análisis de sensibilidad de hiperparámetros\n")

for chunk_size, k, search_type in product(chunk_sizes, k_values, search_types):
    # Recrear splits
    text_splitter_test = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=50
    )
    splits_test = text_splitter_test.split_documents(docs)
    
    # Recrear vectorstore
    vectorstore_test = FAISS.from_documents(documents=splits_test, embedding=embedding)
    
    # Recrear retriever
    retriever_test = vectorstore_test.as_retriever(
        search_type=search_type,
        search_kwargs={"k": k}
    )
    
    # Recrear chain
    rag_chain_test = (
        {
            "context": retriever_test | format_docs,
            "question": RunnablePassthrough(),
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    
    respuesta = rag_chain_test.invoke(pregunta_test)
    
    print(f"\nChunk: {chunk_size}, K: {k}, Search: {search_type}")
    print(f"Respuesta: {respuesta}...")
    print("-" * 80)

Análisis de sensibilidad de hiperparámetros


Chunk: 300, K: 2, Search: similarity
Respuesta: El equilibrio de Nash es una situación en la que los agentes económicos, que interactúan unos con otros, seleccionan su mejor estrategia, dadas las es...
--------------------------------------------------------------------------------

Chunk: 300, K: 2, Search: similarity
Respuesta: El equilibrio de Nash es una situación en la que los agentes económicos, que interactúan unos con otros, seleccionan su mejor estrategia, dadas las es...
--------------------------------------------------------------------------------

Chunk: 300, K: 2, Search: mmr
Respuesta: El equilibrio de Nash es una situación en la que los agentes económicos, que interactúan unos con otros, seleccionan su mejor estrategia, dadas las es...
--------------------------------------------------------------------------------

Chunk: 300, K: 2, Search: mmr
Respuesta: El equilibrio de Nash es una situación en la que los agentes económi

In [ ]:
print(f"\nChunk: {chunk_size}, K: {k}, Search: {search_type}")
print(f"Respuesta: {respuesta}...")
print("-" * 80)

### **2.2 Agentes (1.0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/rcqnN2aJCSEAAAAd/secret-agent-man.gif"
" width="400">
</p>

Similar a la sección anterior, en esta sección se busca habilitar **Agentes** para obtener información a través de tools y así responder la pregunta del usuario.

#### **2.2.1 Tool de Tavily (0.2 puntos)**

Generar una *tool* que pueda hacer consultas al motor de búsqueda **Tavily**.

In [27]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Crear tool de Tavily
tavily_search = TavilySearchResults(max_results=2)
print("Tool de Tavily creada exitosamente")

Tool de Tavily creada exitosamente


/tmp/ipykernel_57286/3404468370.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=2)


In [28]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Crear tool de Wikipedia
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
print("Tool de Wikipedia creada exitosamente")

Tool de Wikipedia creada exitosamente


#### **2.2.2 Tool de Wikipedia (0.2 puntos)**

Generar una *tool* que pueda hacer consultas a **Wikipedia**.

*Hint: Le puede ser de ayuda el siguiente [link](https://python.langchain.com/v0.1/docs/modules/tools/).*

In [ ]:
%pip install --upgrade --quiet wikipedia

#### **2.2.3 Crear Agente (0.3 puntos)**

Crear un agente que pueda responder preguntas preguntas usando las *tools* antes generadas. Asegúrese que su agente responda en español. Por último, guarde el agente en una variable.

**Análisis:**

- **Tavily**: Se usa para información actualizada, noticias recientes, datos en tiempo real. Ideal para preguntas sobre eventos actuales.

- **Wikipedia**: Se usa para conocimiento general, definiciones, información enciclopédica. Ideal para conceptos establecidos y hechos históricos.

In [31]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# Agrupar tools
tools_agent = [tavily_search, wikipedia]

# Template para el agente
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil que puede buscar información en internet. Siempre responde en español."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Crear agente
agent = create_tool_calling_agent(llm, tools_agent, agent_prompt)
agente_busqueda = AgentExecutor(agent=agent, tools=tools_agent, verbose=False)

print("Agente de búsqueda creado exitosamente")

Agente de búsqueda creado exitosamente


#### **2.2.4 Verificación de respuestas (0.3 puntos)**

Pruebe el funcionamiento de su agente y asegúrese que el agente esté ocupando correctamente las tools disponibles. ¿En qué casos el agente debería ocupar la tool de Tavily? ¿En qué casos debería ocupar la tool de Wikipedia?

In [32]:
# Preguntas de prueba
preguntas_agente = [
    "¿Quién ganó las elecciones presidenciales de Estados Unidos en 2024?",
    "¿Qué es el cambio climático?",
    "¿Cuál es la capital de Francia?"
]

for pregunta in preguntas_agente:
    print(f"{'='*80}")
    print(f"PREGUNTA: {pregunta}")
    
    response = agente_busqueda.invoke({"input": pregunta})
    print(f"\nRESPUESTA: {response['output']}")
    print(f"\n{'='*80}")

PREGUNTA: ¿Quién ganó las elecciones presidenciales de Estados Unidos en 2024?

RESPUESTA: Según los resultados de la búsqueda, Donald Trump ganó las elecciones presidenciales de Estados Unidos en 2024.

PREGUNTA: ¿Qué es el cambio climático?

RESPUESTA: Según los resultados de la búsqueda, Donald Trump ganó las elecciones presidenciales de Estados Unidos en 2024.

PREGUNTA: ¿Qué es el cambio climático?

RESPUESTA: El cambio climático se refiere a las variaciones a largo plazo de las temperaturas y los patrones climáticos. Estas variaciones pueden ser naturales, pero desde el siglo XIX, las actividades humanas han sido el principal motor del cambio climático, principalmente debido a la quema de combustibles fósiles (como el carbón, el petróleo y el gas), que produce gases de efecto invernadero.


PREGUNTA: ¿Cuál es la capital de Francia?

RESPUESTA: El cambio climático se refiere a las variaciones a largo plazo de las temperaturas y los patrones climáticos. Estas variaciones pueden ser

### **2.3 Multi Agente (1.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/r7QMJLxU4BoAAAAd/this-is-getting-out-of-hand-star-wars.gif"
" width="450">
</p>

El objetivo de esta subsección es encapsular las funcionalidades creadas en una solución multiagente con un **supervisor**.


#### **2.3.1 Generando Tools (0.5 puntos)**

Transforme la solución RAG de la sección 2.1 y el agente de la sección 2.2 a *tools* (una tool por cada uno).

In [33]:
from langchain.tools import tool

@tool
def rag_tool(question: str) -> str:
    """Responde preguntas sobre teoría de juegos y economía usando documentos PDF."""
    return rag_chain.invoke(question)

@tool
def search_tool(question: str) -> str:
    """Busca información actualizada en internet sobre cualquier tema."""
    response = agente_busqueda.invoke({"input": question})
    return response['output']

print("Tools creadas: rag_tool y search_tool")

Tools creadas: rag_tool y search_tool


**Análisis:**

El supervisor actúa como un router inteligente que:
- Detecta el tipo de pregunta
- Selecciona la herramienta apropiada (RAG o búsqueda web)
- Coordina la respuesta final

Las respuestas son más consistentes y el sistema es más robusto al combinar ambas fuentes.

#### **2.3.2 Agente Supervisor (0.5 puntos)**

Habilite un agente que tenga acceso a las tools del punto anterior y pueda responder preguntas relacionadas. Almacene este agente en una variable llamada supervisor.

In [34]:
# Tools del supervisor
supervisor_tools = [rag_tool, search_tool]

# Template para supervisor
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente inteligente que coordina diferentes herramientas para responder preguntas.
Siempre responde en español de forma completa y útil.

Tienes acceso a:
- rag_tool: Para preguntas sobre teoría de juegos y economía (basado en PDFs)
- search_tool: Para información actualizada de internet

Debes elegir la herramienta más apropiada según la pregunta del usuario."""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])


# Crear supervisorprint("Supervisor creado exitosamente")

supervisor_agent = create_tool_calling_agent(llm, supervisor_tools, supervisor_prompt)
supervisor = AgentExecutor(agent=supervisor_agent, tools=supervisor_tools, verbose=False)

#### **2.3.3 Verificación de respuestas (0.25 puntos)**

Pruebe el funcionamiento de su agente repitiendo las preguntas realizadas en las secciones 2.1.4 y 2.2.4 y comente sus resultados. ¿Cómo varían las respuestas bajo este enfoque?

In [35]:
# Repetir preguntas de secciones anteriores
preguntas_supervisor = [
    "¿Qué es el equilibrio de Nash?",  # De 2.1.4
    "¿Quién ganó las elecciones presidenciales de Estados Unidos en 2024?",  # De 2.2.4
]

for pregunta in preguntas_supervisor:
    print(f"\n{'='*80}")
    print(f"PREGUNTA: {pregunta}")
    
    response = supervisor.invoke({"input": pregunta})
    print(f"\nRESPUESTA SUPERVISOR: {response['output']}")
    print(f"{'='*80}")


PREGUNTA: ¿Qué es el equilibrio de Nash?

RESPUESTA SUPERVISOR: El equilibrio de Nash es un concepto fundamental en la teoría de juegos. Se define como una situación en la que cada jugador elige la mejor estrategia posible, teniendo en cuenta las estrategias elegidas por los demás jugadores. En otras palabras, ningún jugador tiene incentivos para cambiar su estrategia unilateralmente si todos los demás mantienen las suyas. Es un punto de estabilidad en un juego, donde las expectativas de los jugadores sobre las acciones de los demás se cumplen, y nadie se beneficia de desviarse de su elección actual.


PREGUNTA: ¿Quién ganó las elecciones presidenciales de Estados Unidos en 2024?

RESPUESTA SUPERVISOR: El equilibrio de Nash es un concepto fundamental en la teoría de juegos. Se define como una situación en la que cada jugador elige la mejor estrategia posible, teniendo en cuenta las estrategias elegidas por los demás jugadores. En otras palabras, ningún jugador tiene incentivos para ca

#### **2.3.4 Análisis (0.25 puntos)**

¿Qué diferencias tiene este enfoque con la solución *Router* vista en clases? Nombre al menos una ventaja y desventaja.

**Diferencias con Router:**

- Puede ser más lento que un router simple basado en reglas

**Ventajas del enfoque Supervisor:**
- Mayor flexibilidad: el LLM decide dinámicamente qué tool usar
- Puede combinar múltiples tools en una misma respuesta
- Más adaptable a preguntas ambiguas o complejas
  
**Desventajas del enfoque Supervisor:**

- Menos predecible: la decisión no es determinística
- Más costoso: requiere más llamadas al LLM

### **2.4 Memoria (Bonus +0.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/Gs95aiElrscAAAAd/memory-unlocked-ratatouille-critic.gif"
" width="400">
</p>

Una de las principales falencias de las soluciones que hemos visto hasta ahora es que nuestro chat no responde las interacciones anteriores, por ejemplo:

- Pregunta 1: "Hola! mi nombre es Sebastián"
  - Respuesta esperada: "Hola Sebastián! ..."
- Pregunta 2: "Cual es mi nombre?"
  - Respuesta actual: "Lo siento pero no conozco tu nombre :("
  - **Respuesta esperada: "Tu nombre es Sebastián"**

Para solucionar esto, se les solicita agregar un componente de **memoria** a la solución entregada en el punto 2.3.

**Nota: El Bonus es válido <u>sólo para la sección 2 de Large Language Models.</u>**

In [36]:
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import MessagesPlaceholder

# Memoria para el supervisor
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)

# Prompt del supervisor con historial
supervisor_prompt_memory = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente inteligente que coordina diferentes herramientas para responder preguntas.
Recuerdas conversaciones anteriores.

Tienes acceso a:
- rag_tool: Para preguntas sobre teoría de juegos y economía (basado en PDFs)
- search_tool: Para información actualizada en internet

Debes elegir la herramienta más apropiada según la pregunta del usuario.
Siempre responde en español de forma completa y útil."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Agente y ejecutor con memoria
supervisor_agent = create_tool_calling_agent(llm, supervisor_tools, supervisor_prompt_memory)
supervisor = AgentExecutor(
    agent=supervisor_agent,
    tools=supervisor_tools,
    memory=memory,
    verbose=False,
)

# Función wrapper para usar el supervisor con memoria

def supervisor_con_memoria(pregunta: str) -> str:
    response = supervisor.invoke({"input": pregunta})
    return response["output"]

# Prueba de memoria
print("Pregunta 1: Hola! mi nombre es Javier")
resp1 = supervisor_con_memoria("Hola! mi nombre es Javier")
print(f"Respuesta: {resp1}
")

print("Pregunta 2: ¿Cuál es mi nombre?")
resp2 = supervisor_con_memoria("¿Cuál es mi nombre?")
print(f"Respuesta: {resp2}")



Pregunta 1: Hola! mi nombre es Javier


/tmp/ipykernel_57286/539540404.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


Respuesta: Hola Javier, ¡un gusto conocerte! ¿En qué puedo ayudarte hoy?


Pregunta 2: ¿Cuál es mi nombre?
Respuesta: Como un modelo de lenguaje, no tengo nombre.
Respuesta: Como un modelo de lenguaje, no tengo nombre.


### **2.5 Despliegue (0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/IytHqOp52EsAAAAd/you-get-a-deploy-deploy.gif"
" width="400">
</p>

Una vez tengan los puntos anteriores finalizados, toca la etapa de dar a conocer lo que hicimos! Para eso, vamos a desplegar nuestro modelo a través de `gradio`, una librería especializada en el levantamiento rápido de demos basadas en ML.

Primero instalamos la librería:

In [ ]:
%pip install --upgrade --quiet gradio

Luego sólo deben ejecutar el siguiente código e interactuar con la interfaz a través del notebook o del link generado:

In [37]:
import gradio as gr
import time

def agent_response(message, history):
    '''
    Función para gradio, recibe mensaje e historial, devuelte la respuesta del chatbot.
    '''
    # Obtener respuesta del supervisor con memoria
    response = supervisor_con_memoria(message)
    
    # Assert
    assert type(response) == str, "output debe ser string"
    
    # "streaming" response
    for i in range(len(response)):
        time.sleep(0.015)
        yield response[: i+1]

gr.ChatInterface(
    agent_response,
    type="messages",
    title="Chatbot MDS7202 - Teoría de Juegos y Economía",
    description="¡Hola! Puedo responder preguntas sobre teoría de juegos, economía y buscar información actualizada en internet.",
    theme="soft",
).launch(
    share=True,
    debug=False,
)

/home/javi02/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://58453c65a4ac8db41c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
* Running on public URL: https://58453c65a4ac8db41c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Conclusión
Éxito!
<center>
<img src ="https://media.tenor.com/MRQgxcelAV8AAAAM/perry-the-platypus-phineas-and-ferb.gif" width = 400 />